# Tutorial: End-to-End Colab Fine-Tuning for Text-to-SQL with Unsloth

Audience:
- You want one Colab notebook that can run the whole fine-tuning pipeline for Text-to-SQL.
- You want the same notebook to support a 20-row smoke test and a 5,000-row run.

Prerequisites:
- A GPU Colab runtime. T4 is enough for the 20-row smoke test. For 5k rows, prefer L4 or A100.
- Optional: `/content/Model-FineTuning.zip` if you want to start from your uploaded project archive.
- Optional: a Hugging Face token if your runtime needs authenticated model access.

Learning goals:
- Bootstrap the project automatically in Colab.
- Rewrite the core files to the current Colab-compatible versions.
- Change one parameter to switch between 20 rows and 5,000 rows.
- Fine-tune `Qwen2.5-Coder-7B-Instruct` with Unsloth QLoRA.
- Run a stable post-training inference check and package the adapter.


## Outline

1. Set the run parameters
2. Bootstrap or create the project in Colab
3. Sync the Colab-compatible project files
4. Install dependencies
5. Generate a runtime config from `DATASET_ROWS`
6. Inspect and preprocess the Gretel dataset
7. Train with Unsloth
8. Run post-training inference with `transformers + peft`
9. Zip the adapter and optionally copy artifacts to Drive


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/Model-FineTuning')
ZIP_PATH = Path('/content/Model-FineTuning.zip')
REPO_URL = ''  # Optional fallback if you want to clone instead of uploading a zip.
HF_TOKEN = ''  # Optional. Set only if HF access requires auth.
SPIDER_BLOCKLIST_JSON = ''  # Optional path, for example /content/spider_dev_blocklist.json

DATASET_ROWS = 20  # Change this to 5000 for the scale-up run.
BASE_MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
TRANSFORMERS_BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
RUN_NAME = f'colab_{DATASET_ROWS}_qwen25_coder'

MAX_SEQ_LENGTH = 1536 if DATASET_ROWS <= 100 else 2048
NUM_EPOCHS = 10 if DATASET_ROWS <= 20 else (3 if DATASET_ROWS <= 100 else 1)
TRAIN_BATCH_SIZE = 1 if DATASET_ROWS <= 100 else 2
EVAL_BATCH_SIZE = TRAIN_BATCH_SIZE
GRAD_ACCUM = 4 if DATASET_ROWS <= 100 else 8
LEARNING_RATE = 2e-4 if DATASET_ROWS <= 100 else 1e-4
LORA_R = 16 if DATASET_ROWS <= 100 else 32
LORA_ALPHA = 32 if DATASET_ROWS <= 100 else 64
EXPORT_MERGED_16BIT = False  # Set True only on stronger GPUs if you need a merged HF model.
RUN_PREVIEW_INFERENCE = True
SAVE_TO_DRIVE = False
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/text2sql_unsloth'


def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f'\n$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        executable='/bin/bash',
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return result

print({
    'DATASET_ROWS': DATASET_ROWS,
    'RUN_NAME': RUN_NAME,
    'MAX_SEQ_LENGTH': MAX_SEQ_LENGTH,
    'NUM_EPOCHS': NUM_EPOCHS,
    'TRAIN_BATCH_SIZE': TRAIN_BATCH_SIZE,
    'GRAD_ACCUM': GRAD_ACCUM,
    'LEARNING_RATE': LEARNING_RATE,
    'LORA_R': LORA_R,
    'LORA_ALPHA': LORA_ALPHA,
    'EXPORT_MERGED_16BIT': EXPORT_MERGED_16BIT,
})


## Step 1 - Bootstrap the project directory

This cell works in four modes:
- use `/content/Model-FineTuning` if it already exists
- unzip `/content/Model-FineTuning.zip` if present
- clone from `REPO_URL` if set
- otherwise create a fresh project directory and let the notebook write the files itself


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

if PROJECT_DIR.exists():
    print(f'Using existing project at {PROJECT_DIR}')
elif ZIP_PATH.exists():
    print(f'Found uploaded zip at {ZIP_PATH}. Extracting to /content ...')
    run(f'unzip -o {ZIP_PATH} -d /content')
    nested_dir = PROJECT_DIR / 'Model-FineTuning'
    if nested_dir.exists() and not (PROJECT_DIR / 'scripts').exists():
        print('Detected nested Model-FineTuning directory. Flattening it ...')
        for item in nested_dir.iterdir():
            target = PROJECT_DIR / item.name
            if target.exists():
                continue
            shutil.move(str(item), str(target))
        shutil.rmtree(nested_dir)
elif REPO_URL:
    run(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Created fresh project directory at {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())


## Step 2 - Sync the Colab-compatible project files

This makes the notebook self-healing. Even if the uploaded zip is older, the core training and preprocessing files are overwritten with the current versions before installation.


In [ ]:
from pathlib import Path

FILES = {
    "pyproject.toml": "[build-system]\nrequires = [\"setuptools>=68\", \"wheel\"]\nbuild-backend = \"setuptools.build_meta\"\n\n[project]\nname = \"text2sql-unsloth\"\nversion = \"0.1.0\"\ndescription = \"Reproducible Text-to-SQL fine-tuning pipeline with Unsloth and QLoRA.\"\nreadme = \"README.md\"\nrequires-python = \">=3.10\"\ndependencies = []\n\n[tool.setuptools]\npackage-dir = {\"\" = \"src\"}\n\n[tool.setuptools.packages.find]\nwhere = [\"src\"]\n\n",
    "requirements/common.txt": "accelerate>=1.2.1\nbitsandbytes>=0.45.0\ndatasets>=3.2.0\nnumpy>=1.26.0\npackaging>=24.2\npeft>=0.14.0\npydantic>=2.10.0\npyyaml>=6.0.2\nsqlglot>=26.0.0\ntorch>=2.5.1\ntransformers>=4.49.0\ntrl>=0.13.0\nunsloth @ git+https://github.com/unslothai/unsloth.git\nunsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git\n\n",
    "requirements/colab.txt": "-r common.txt\njupyter>=1.1.1\n\n",
    "configs/base.yaml": "seed: 3407\n\nmodel:\n  base_model_name: \"Qwen/Qwen2.5-Coder-7B-Instruct\"\n  load_in_4bit: true\n  max_seq_length: 2048\n  dtype: null\n  trust_remote_code: true\n  lora:\n    r: 32\n    alpha: 64\n    dropout: 0.0\n    bias: \"none\"\n    use_rslora: false\n    use_gradient_checkpointing: \"unsloth\"\n    target_modules:\n      - \"q_proj\"\n      - \"k_proj\"\n      - \"v_proj\"\n      - \"o_proj\"\n      - \"gate_proj\"\n      - \"up_proj\"\n      - \"down_proj\"\n\ndataset:\n  hf_name: \"gretelai/synthetic_text_to_sql\"\n  hf_split: \"train\"\n  sample_size: null\n  output_root: \"data/processed/default\"\n  include_explanation: false\n  allowed_task_types:\n    - \"analytics and reporting\"\n  allowed_leading_keywords:\n    - \"SELECT\"\n    - \"WITH\"\n  blocked_sql_keywords:\n    - \"INSERT\"\n    - \"UPDATE\"\n    - \"DELETE\"\n    - \"DROP\"\n    - \"ALTER\"\n    - \"CREATE\"\n    - \"REPLACE\"\n    - \"TRUNCATE\"\n    - \"MERGE\"\n    - \"GRANT\"\n    - \"REVOKE\"\n  blocked_dialect_regex:\n    - \"\\\\bDATE_SUB\\\\s*\\\\(\"\n    - \"\\\\bCURDATE\\\\s*\\\\(\"\n    - \"\\\\bNOW\\\\s*\\\\(\"\n    - \"\\\\bSYSDATE\\\\b\"\n    - \"\\\\bDATEDIFF\\\\s*\\\\(\"\n    - \"\\\\bDATEADD\\\\s*\\\\(\"\n    - \"\\\\bTIMESTAMPDIFF\\\\s*\\\\(\"\n    - \"\\\\bINTERVAL\\\\b\"\n    - \"\\\\bILIKE\\\\b\"\n    - \"\\\\bQUALIFY\\\\b\"\n    - \"\\\\bUNNEST\\\\b\"\n    - \"\\\\bSTRING_AGG\\\\s*\\\\(\"\n    - \"\\\\bLISTAGG\\\\s*\\\\(\"\n    - \"\\\\bARRAY\\\\s*\\\\[\"\n    - \"\\\\bSTRUCT\\\\s*\\\\(\"\n    - \"\\\\bTOP\\\\s+[0-9]+\"\n    - \"\\\\bCROSS\\\\s+APPLY\\\\b\"\n    - \"\\\\bOUTER\\\\s+APPLY\\\\b\"\n    - \"\\\\bTRY_CAST\\\\s*\\\\(\"\n    - \"\\\\bREGEXP\\\\b\"\n    - \"\\\\bEXTRACT\\\\s*\\\\(\"\n  min_prompt_chars: 12\n  min_sql_chars: 8\n  require_schema_context: true\n  keep_create_view: false\n  dedupe_on:\n    - \"sql_prompt\"\n    - \"schema_ddl\"\n    - \"sql\"\n  split:\n    train_ratio: 0.90\n    val_ratio: 0.05\n    test_ratio: 0.05\n\nprompt:\n  train_format: \"direct_sql\"\n  inference_format: \"advanced_reasoning\"\n  direct_sql_system_prompt: |\n    You are an expert Text-to-SQL assistant.\n    Convert each question into executable SQLite SQL using only the provided schema.\n    Return SQL only.\n    Do not return explanations, comments, markdown, or code fences.\n  advanced_reasoning_system_prompt: |\n    You are an expert SQLite Text-to-SQL assistant.\n    Work through schema selection, join paths, filters, aggregation, and ordering internally.\n    Return only the final executable SQLite SQL.\n    Do not include explanations, comments, markdown, or code fences.\n  user_template: |\n    Database schema:\n    {schema}\n\n    Question:\n    {question}\n\n    Return only SQLite SQL.\n\ntraining:\n  output_dir: \"artifacts/qwen25_coder_text2sql\"\n  per_device_train_batch_size: 2\n  per_device_eval_batch_size: 2\n  gradient_accumulation_steps: 8\n  learning_rate: 1.0e-4\n  lr_scheduler_type: \"cosine\"\n  warmup_ratio: 0.05\n  weight_decay: 0.01\n  max_grad_norm: 0.3\n  num_train_epochs: 1\n  max_steps: null\n  logging_steps: 5\n  eval_steps: 50\n  save_steps: 50\n  save_total_limit: 2\n  optim: \"adamw_8bit\"\n  gradient_checkpointing: true\n  group_by_length: true\n  dataloader_num_workers: 0\n  report_to: \"none\"\n\nexport:\n  save_adapter: true\n  save_merged_16bit: false\n  save_gguf: false\n  gguf_quantization_method: \"q4_k_m\"\n\nruntime:\n  dataset_num_proc: 1\n\n",
    "src/text2sql_unsloth/__init__.py": "\"\"\"Utilities for the Text-to-SQL Unsloth pipeline.\"\"\"\n\n",
    "src/text2sql_unsloth/config.py": "from __future__ import annotations\n\nimport copy\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n\ndef load_yaml(path: str | Path) -> dict[str, Any]:\n    with Path(path).open(\"r\", encoding=\"utf-8\") as handle:\n        return yaml.safe_load(handle) or {}\n\n\ndef deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:\n    merged = copy.deepcopy(base)\n    for key, value in override.items():\n        if isinstance(value, dict) and isinstance(merged.get(key), dict):\n            merged[key] = deep_update(merged[key], value)\n        else:\n            merged[key] = value\n    return merged\n\n\ndef load_config(base_path: str | Path, override_path: str | Path | None = None) -> dict[str, Any]:\n    config = load_yaml(base_path)\n    if override_path:\n        config = deep_update(config, load_yaml(override_path))\n    return config\n\n\ndef ensure_dir(path: str | Path) -> Path:\n    resolved = Path(path)\n    resolved.mkdir(parents=True, exist_ok=True)\n    return resolved\n\n",
    "src/text2sql_unsloth/prompting.py": "from __future__ import annotations\n\nfrom typing import Any\n\n\ndef build_user_message(question: str, schema: str, user_template: str) -> str:\n    return user_template.format(\n        question=question.strip(),\n        schema=schema.strip(),\n    ).strip()\n\n\ndef build_messages(\n    *,\n    question: str,\n    schema: str,\n    sql: str,\n    config: dict[str, Any],\n    include_explanation: bool = False,\n    sql_explanation: str | None = None,\n    format_name: str = \"direct_sql\",\n) -> list[dict[str, str]]:\n    prompt_cfg = config[\"prompt\"]\n    if format_name == \"advanced_reasoning\":\n        system_prompt = prompt_cfg[\"advanced_reasoning_system_prompt\"].strip()\n    else:\n        system_prompt = prompt_cfg[\"direct_sql_system_prompt\"].strip()\n\n    user_content = build_user_message(\n        question=question,\n        schema=schema,\n        user_template=prompt_cfg[\"user_template\"],\n    )\n    assistant_content = sql.strip()\n    if include_explanation and sql_explanation:\n        assistant_content = (\n            \"Reasoning:\\n\"\n            f\"{sql_explanation.strip()}\\n\\n\"\n            \"Final SQL:\\n\"\n            f\"{sql.strip()}\"\n        )\n\n    return [\n        {\"role\": \"system\", \"content\": system_prompt},\n        {\"role\": \"user\", \"content\": user_content},\n        {\"role\": \"assistant\", \"content\": assistant_content},\n    ]\n\n\ndef render_chat(tokenizer: Any, messages: list[dict[str, str]], *, add_generation_prompt: bool) -> str:\n    return tokenizer.apply_chat_template(\n        messages,\n        tokenize=False,\n        add_generation_prompt=add_generation_prompt,\n    )\n\n\ndef extract_sql_from_response(text: str) -> str:\n    stripped = text.strip()\n    if \"```sql\" in stripped:\n        after = stripped.split(\"```sql\", 1)[1]\n        return after.split(\"```\", 1)[0].strip()\n    if \"```\" in stripped:\n        after = stripped.split(\"```\", 1)[1]\n        return after.split(\"```\", 1)[0].strip()\n\n    final_sql_prefix = \"Final SQL:\"\n    if final_sql_prefix in stripped:\n        return stripped.split(final_sql_prefix, 1)[1].strip()\n    return stripped\n\n",
    "src/text2sql_unsloth/sql_filters.py": "from __future__ import annotations\n\nimport json\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport sqlglot\n\n\nWHITESPACE_RE = re.compile(r\"\\s+\")\n\n\ndef normalize_whitespace(text: str) -> str:\n    return WHITESPACE_RE.sub(\" \", text or \"\").strip()\n\n\ndef normalize_text_key(text: str) -> str:\n    return normalize_whitespace(text).lower()\n\n\ndef normalize_sql_for_dedupe(sql: str) -> str:\n    sql = normalize_whitespace(sql)\n    return sql.rstrip(\";\").lower()\n\n\ndef split_sql_statements(sql_text: str) -> list[str]:\n    statements: list[str] = []\n    current: list[str] = []\n    quote: str | None = None\n    escaped = False\n\n    for char in sql_text:\n        current.append(char)\n        if escaped:\n            escaped = False\n            continue\n        if char == \"\\\\\":\n            escaped = True\n            continue\n        if quote:\n            if char == quote:\n                quote = None\n            continue\n        if char in {\"'\", '\"'}:\n            quote = char\n            continue\n        if char == \";\":\n            statement = \"\".join(current).strip()\n            if statement:\n                statements.append(statement)\n            current = []\n\n    tail = \"\".join(current).strip()\n    if tail:\n        statements.append(tail)\n    return statements\n\n\ndef extract_schema_ddl(sql_context: str, *, keep_create_view: bool = False) -> str:\n    statements = []\n    for statement in split_sql_statements(sql_context):\n        stripped = statement.strip().rstrip(\";\")\n        upper = stripped.upper()\n        if upper.startswith(\"CREATE TABLE\"):\n            statements.append(f\"{stripped};\")\n        elif keep_create_view and upper.startswith(\"CREATE VIEW\"):\n            statements.append(f\"{stripped};\")\n    return \"\\n\".join(statements).strip()\n\n\ndef leading_keyword(sql: str) -> str:\n    normalized = normalize_whitespace(sql).lstrip(\"(\")\n    match = re.match(r\"([A-Za-z]+)\", normalized)\n    return (match.group(1) if match else \"\").upper()\n\n\ndef is_read_only_sql(sql: str, allowed_leading_keywords: list[str], blocked_keywords: list[str]) -> bool:\n    upper = sql.upper()\n    if leading_keyword(sql) not in set(allowed_leading_keywords):\n        return False\n    return not any(re.search(rf\"\\b{re.escape(keyword)}\\b\", upper) for keyword in blocked_keywords)\n\n\ndef has_blocklisted_dialect(sql: str, patterns: list[str]) -> bool:\n    return any(re.search(pattern, sql, flags=re.IGNORECASE) for pattern in patterns)\n\n\ndef parse_sqlite(sql: str) -> bool:\n    try:\n        sqlglot.parse_one(sql, read=\"sqlite\")\n        return True\n    except Exception:\n        return False\n\n\ndef canonicalize_sqlite_query(sql: str) -> str:\n    expression = sqlglot.parse_one(sql, read=\"sqlite\")\n    return expression.sql(dialect=\"sqlite\", pretty=False)\n\n\ndef canonicalize_schema_ddl(schema_ddl: str) -> str:\n    canonicalized: list[str] = []\n    for statement in split_sql_statements(schema_ddl):\n        stripped = statement.strip().rstrip(\";\")\n        if not stripped:\n            continue\n        try:\n            expression = sqlglot.parse_one(stripped, read=\"sqlite\")\n            canonicalized.append(expression.sql(dialect=\"sqlite\", pretty=False) + \";\")\n        except Exception:\n            canonicalized.append(normalize_whitespace(stripped) + \";\")\n    return \"\\n\".join(canonicalized).strip()\n\n\ndef load_spider_blocklist(path: str | Path | None) -> dict[str, set[str]]:\n    if not path:\n        return {\"questions\": set(), \"sql\": set(), \"pairs\": set()}\n\n    data_path = Path(path)\n    raw = json.loads(data_path.read_text(encoding=\"utf-8\"))\n    questions: set[str] = set()\n    sql_entries: set[str] = set()\n    pairs: set[str] = set()\n\n    for item in raw:\n        question = normalize_text_key(str(item.get(\"question\") or item.get(\"prompt\") or \"\"))\n        sql_value = normalize_sql_for_dedupe(str(item.get(\"sql\") or item.get(\"query\") or \"\"))\n        if question:\n            questions.add(question)\n        if sql_value:\n            sql_entries.add(sql_value)\n        if question and sql_value:\n            pairs.add(f\"{question}|||{sql_value}\")\n    return {\"questions\": questions, \"sql\": sql_entries, \"pairs\": pairs}\n\n\ndef in_spider_blocklist(question: str, sql: str, blocklist: dict[str, set[str]]) -> bool:\n    normalized_question = normalize_text_key(question)\n    normalized_sql = normalize_sql_for_dedupe(sql)\n    return (\n        normalized_question in blocklist[\"questions\"]\n        or normalized_sql in blocklist[\"sql\"]\n        or f\"{normalized_question}|||{normalized_sql}\" in blocklist[\"pairs\"]\n    )\n\n\ndef build_dedupe_key(item: dict[str, Any], fields: list[str]) -> str:\n    pieces = []\n    for field in fields:\n        value = item.get(field, \"\")\n        if field == \"sql\":\n            pieces.append(normalize_sql_for_dedupe(str(value)))\n        else:\n            pieces.append(normalize_text_key(str(value)))\n    return \"|||\".join(pieces)\n",
    "scripts/preprocess_gretel.py": "from __future__ import annotations\n\nimport argparse\nimport json\nimport random\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\nfrom datasets import load_dataset\n\nfrom text2sql_unsloth.config import ensure_dir, load_config\nfrom text2sql_unsloth.prompting import build_messages\nfrom text2sql_unsloth.sql_filters import (\n    build_dedupe_key,\n    canonicalize_schema_ddl,\n    canonicalize_sqlite_query,\n    extract_schema_ddl,\n    has_blocklisted_dialect,\n    in_spider_blocklist,\n    is_read_only_sql,\n    load_spider_blocklist,\n    normalize_whitespace,\n    parse_sqlite,\n)\n\n\nREQUIRED_COLUMNS = {\n    \"domain\",\n    \"domain_description\",\n    \"sql_complexity\",\n    \"sql_task_type\",\n    \"sql_prompt\",\n    \"sql_context\",\n    \"sql\",\n    \"sql_explanation\",\n}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Preprocess Gretel synthetic Text-to-SQL data.\")\n    parser.add_argument(\"--base-config\", default=\"configs/base.yaml\")\n    parser.add_argument(\"--config\", default=None, help=\"Optional override config.\")\n    parser.add_argument(\"--sample-size\", type=int, default=None)\n    parser.add_argument(\"--seed\", type=int, default=None)\n    parser.add_argument(\"--output-root\", default=None)\n    parser.add_argument(\"--spider-blocklist-json\", default=None)\n    parser.add_argument(\"--inspect-only\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef describe_dataset(records: list[dict[str, Any]]) -> dict[str, Any]:\n    task_counts = Counter(record[\"sql_task_type\"] for record in records)\n    complexity_counts = Counter(record[\"sql_complexity\"] for record in records)\n    return {\n        \"row_count\": len(records),\n        \"task_type_counts\": dict(task_counts.most_common()),\n        \"sql_complexity_counts\": dict(complexity_counts.most_common()),\n    }\n\n\ndef clean_records(\n    records: list[dict[str, Any]],\n    config: dict[str, Any],\n    spider_blocklist: dict[str, set[str]],\n) -> tuple[list[dict[str, Any]], dict[str, int]]:\n    dataset_cfg = config[\"dataset\"]\n    include_explanation = bool(dataset_cfg.get(\"include_explanation\", False))\n    stats = Counter()\n    cleaned: list[dict[str, Any]] = []\n\n    for raw in records:\n        stats[\"seen\"] += 1\n\n        prompt = normalize_whitespace(str(raw.get(\"sql_prompt\", \"\")))\n        sql = normalize_whitespace(str(raw.get(\"sql\", \"\")))\n        sql_context = str(raw.get(\"sql_context\", \"\") or \"\")\n        task_type = str(raw.get(\"sql_task_type\", \"\") or \"\")\n\n        if len(prompt) < dataset_cfg[\"min_prompt_chars\"]:\n            stats[\"drop_short_prompt\"] += 1\n            continue\n        if len(sql) < dataset_cfg[\"min_sql_chars\"]:\n            stats[\"drop_short_sql\"] += 1\n            continue\n        if task_type not in set(dataset_cfg[\"allowed_task_types\"]):\n            stats[\"drop_task_type\"] += 1\n            continue\n        if not is_read_only_sql(\n            sql,\n            allowed_leading_keywords=dataset_cfg[\"allowed_leading_keywords\"],\n            blocked_keywords=dataset_cfg[\"blocked_sql_keywords\"],\n        ):\n            stats[\"drop_non_read_only_sql\"] += 1\n            continue\n        if has_blocklisted_dialect(sql, dataset_cfg[\"blocked_dialect_regex\"]):\n            stats[\"drop_blocklisted_dialect\"] += 1\n            continue\n        if not parse_sqlite(sql):\n            stats[\"drop_sqlite_parse_failed\"] += 1\n            continue\n\n        schema_ddl = extract_schema_ddl(\n            sql_context,\n            keep_create_view=bool(dataset_cfg.get(\"keep_create_view\", False)),\n        )\n        if dataset_cfg.get(\"require_schema_context\", True) and not schema_ddl:\n            stats[\"drop_missing_schema_ddl\"] += 1\n            continue\n\n        canonical_sql = canonicalize_sqlite_query(sql)\n        canonical_schema = canonicalize_schema_ddl(schema_ddl) if schema_ddl else \"\"\n\n        if spider_blocklist and in_spider_blocklist(prompt, canonical_sql, spider_blocklist):\n            stats[\"drop_spider_blocklist\"] += 1\n            continue\n\n        item = {\n            \"domain\": raw.get(\"domain\"),\n            \"domain_description\": raw.get(\"domain_description\"),\n            \"sql_complexity\": raw.get(\"sql_complexity\"),\n            \"sql_task_type\": task_type,\n            \"sql_prompt\": prompt,\n            \"sql_context_raw\": sql_context.strip(),\n            \"schema_ddl\": canonical_schema,\n            \"sql\": canonical_sql,\n            \"sql_explanation\": normalize_whitespace(str(raw.get(\"sql_explanation\", \"\"))),\n        }\n        item[\"messages\"] = build_messages(\n            question=item[\"sql_prompt\"],\n            schema=item[\"schema_ddl\"],\n            sql=item[\"sql\"],\n            sql_explanation=item[\"sql_explanation\"],\n            include_explanation=include_explanation,\n            config=config,\n            format_name=config[\"prompt\"][\"train_format\"],\n        )\n        cleaned.append(item)\n        stats[\"kept_before_dedupe\"] += 1\n\n    dedupe_fields = dataset_cfg[\"dedupe_on\"]\n    deduped: list[dict[str, Any]] = []\n    seen_keys: set[str] = set()\n    for item in cleaned:\n        dedupe_key = build_dedupe_key(item, dedupe_fields)\n        if dedupe_key in seen_keys:\n            stats[\"drop_duplicate\"] += 1\n            continue\n        seen_keys.add(dedupe_key)\n        deduped.append(item)\n\n    stats[\"kept_after_dedupe\"] = len(deduped)\n    return deduped, dict(stats)\n\n\ndef split_records(records: list[dict[str, Any]], config: dict[str, Any], seed: int) -> dict[str, list[dict[str, Any]]]:\n    split_cfg = config[\"dataset\"][\"split\"]\n    shuffled = records[:]\n    random.Random(seed).shuffle(shuffled)\n\n    train_ratio = float(split_cfg[\"train_ratio\"])\n    val_ratio = float(split_cfg[\"val_ratio\"])\n    test_ratio = float(split_cfg[\"test_ratio\"])\n    ratio_sum = train_ratio + val_ratio + test_ratio\n    if abs(ratio_sum - 1.0) > 1e-6:\n        raise ValueError(f\"Split ratios must sum to 1.0, got {ratio_sum}\")\n\n    total = len(shuffled)\n    train_end = int(total * train_ratio)\n    val_end = train_end + int(total * val_ratio)\n    return {\n        \"train\": shuffled[:train_end],\n        \"val\": shuffled[train_end:val_end],\n        \"test\": shuffled[val_end:],\n    }\n\n\ndef write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:\n    with path.open(\"w\", encoding=\"utf-8\") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=True) + \"\\n\")\n\n\ndef main() -> None:\n    args = parse_args()\n    config = load_config(args.base_config, args.config)\n    if args.sample_size is not None:\n        config[\"dataset\"][\"sample_size\"] = args.sample_size\n    if args.output_root is not None:\n        config[\"dataset\"][\"output_root\"] = args.output_root\n    if args.seed is not None:\n        config[\"seed\"] = args.seed\n\n    dataset_cfg = config[\"dataset\"]\n    seed = int(config[\"seed\"])\n\n    dataset = load_dataset(dataset_cfg[\"hf_name\"], split=dataset_cfg[\"hf_split\"])\n    if not REQUIRED_COLUMNS.issubset(set(dataset.column_names)):\n        missing = REQUIRED_COLUMNS - set(dataset.column_names)\n        raise ValueError(f\"Dataset schema changed. Missing columns: {sorted(missing)}\")\n\n    records = list(dataset)\n    summary_before = describe_dataset(records)\n\n    if args.inspect_only:\n        print(json.dumps({\n            \"columns\": dataset.column_names,\n            \"summary_before_filtering\": summary_before,\n            \"example\": {key: records[0][key] for key in sorted(REQUIRED_COLUMNS)},\n        }, indent=2, ensure_ascii=False))\n        return\n\n    spider_blocklist = load_spider_blocklist(args.spider_blocklist_json)\n    cleaned, filter_stats = clean_records(records, config, spider_blocklist)\n\n    sample_size = dataset_cfg.get(\"sample_size\")\n    if sample_size:\n        random.Random(seed).shuffle(cleaned)\n        cleaned = cleaned[: int(sample_size)]\n\n    split_rows = split_records(cleaned, config, seed)\n    output_root = ensure_dir(dataset_cfg[\"output_root\"])\n\n    write_jsonl(output_root / \"all_cleaned.jsonl\", cleaned)\n    for split_name, rows in split_rows.items():\n        write_jsonl(output_root / f\"{split_name}.jsonl\", rows)\n\n    summary = {\n        \"dataset_name\": dataset_cfg[\"hf_name\"],\n        \"split\": dataset_cfg[\"hf_split\"],\n        \"columns\": dataset.column_names,\n        \"summary_before_filtering\": summary_before,\n        \"filter_stats\": filter_stats,\n        \"sample_size\": len(cleaned),\n        \"split_sizes\": {key: len(value) for key, value in split_rows.items()},\n        \"seed\": seed,\n    }\n    (output_root / \"summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n\n",
    "scripts/train_unsloth.py": "from __future__ import annotations\n\nimport argparse\nimport inspect\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport unsloth\nimport torch\nfrom datasets import Dataset, load_dataset\nfrom transformers import Trainer, TrainingArguments\n\nfrom text2sql_unsloth.config import ensure_dir, load_config\nfrom text2sql_unsloth.prompting import render_chat\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Train Qwen2.5-Coder-7B-Instruct with Unsloth QLoRA.\")\n    parser.add_argument(\"--base-config\", default=\"configs/base.yaml\")\n    parser.add_argument(\"--config\", default=None)\n    parser.add_argument(\"--train-file\", default=None)\n    parser.add_argument(\"--val-file\", default=None)\n    parser.add_argument(\"--output-dir\", default=None)\n    return parser.parse_args()\n\n\ndef load_splits(config: dict[str, Any], train_file: str | None, val_file: str | None) -> tuple[Dataset, Dataset]:\n    dataset_root = Path(config[\"dataset\"][\"output_root\"])\n    train_path = train_file or str(dataset_root / \"train.jsonl\")\n    val_path = val_file or str(dataset_root / \"val.jsonl\")\n    data_files = {\"train\": train_path}\n    if Path(val_path).exists():\n        data_files[\"validation\"] = val_path\n    dataset_dict = load_dataset(\"json\", data_files=data_files)\n    train_dataset = dataset_dict[\"train\"]\n    eval_dataset = dataset_dict.get(\"validation\", dataset_dict[\"train\"].select(range(min(8, len(dataset_dict[\"train\"])))))\n    return train_dataset, eval_dataset\n\n\ndef tokenize_example(example: dict[str, Any], tokenizer: Any, max_seq_length: int) -> dict[str, Any]:\n    messages = example[\"messages\"]\n    prompt_messages = messages[:-1]\n\n    full_text = render_chat(tokenizer, messages, add_generation_prompt=False)\n    prompt_text = render_chat(tokenizer, prompt_messages, add_generation_prompt=True)\n\n    full_tokens = tokenizer(\n        full_text,\n        add_special_tokens=False,\n        truncation=True,\n        max_length=max_seq_length,\n    )\n    prompt_tokens = tokenizer(\n        prompt_text,\n        add_special_tokens=False,\n        truncation=True,\n        max_length=max_seq_length,\n    )\n\n    prompt_len = len(prompt_tokens[\"input_ids\"])\n    input_ids = full_tokens[\"input_ids\"]\n    labels = list(input_ids)\n\n    if prompt_len >= len(labels):\n        return {\"skip_example\": True}\n\n    labels[:prompt_len] = [-100] * prompt_len\n    return {\n        \"input_ids\": input_ids,\n        \"attention_mask\": full_tokens[\"attention_mask\"],\n        \"labels\": labels,\n        \"skip_example\": False,\n    }\n\n\nclass SupervisedDataCollator:\n    def __init__(self, tokenizer: Any) -> None:\n        self.tokenizer = tokenizer\n\n    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:\n        max_length = max(len(feature[\"input_ids\"]) for feature in features)\n        pad_id = self.tokenizer.pad_token_id\n\n        input_ids = []\n        attention_masks = []\n        labels = []\n        for feature in features:\n            pad_length = max_length - len(feature[\"input_ids\"])\n            input_ids.append(feature[\"input_ids\"] + [pad_id] * pad_length)\n            attention_masks.append(feature[\"attention_mask\"] + [0] * pad_length)\n            labels.append(feature[\"labels\"] + [-100] * pad_length)\n\n        return {\n            \"input_ids\": torch.tensor(input_ids, dtype=torch.long),\n            \"attention_mask\": torch.tensor(attention_masks, dtype=torch.long),\n            \"labels\": torch.tensor(labels, dtype=torch.long),\n        }\n\n\ndef save_run_manifest(config: dict[str, Any], output_dir: Path) -> None:\n    (output_dir / \"run_config.json\").write_text(json.dumps(config, indent=2), encoding=\"utf-8\")\n\n\ndef main() -> None:\n    args = parse_args()\n    config = load_config(args.base_config, args.config)\n    if args.output_dir:\n        config[\"training\"][\"output_dir\"] = args.output_dir\n\n    model_cfg = config[\"model\"]\n    train_cfg = config[\"training\"]\n    output_dir = ensure_dir(train_cfg[\"output_dir\"])\n    save_run_manifest(config, output_dir)\n\n    from unsloth import FastLanguageModel, is_bfloat16_supported\n\n    model, tokenizer = FastLanguageModel.from_pretrained(\n        model_name=model_cfg[\"base_model_name\"],\n        max_seq_length=model_cfg[\"max_seq_length\"],\n        dtype=model_cfg.get(\"dtype\"),\n        load_in_4bit=bool(model_cfg[\"load_in_4bit\"]),\n    )\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"right\"\n\n    lora_cfg = model_cfg[\"lora\"]\n    model = FastLanguageModel.get_peft_model(\n        model,\n        r=int(lora_cfg[\"r\"]),\n        target_modules=list(lora_cfg[\"target_modules\"]),\n        lora_alpha=int(lora_cfg[\"alpha\"]),\n        lora_dropout=float(lora_cfg[\"dropout\"]),\n        bias=str(lora_cfg[\"bias\"]),\n        use_gradient_checkpointing=lora_cfg[\"use_gradient_checkpointing\"],\n        random_state=int(config[\"seed\"]),\n        max_seq_length=int(model_cfg[\"max_seq_length\"]),\n        use_rslora=bool(lora_cfg.get(\"use_rslora\", False)),\n    )\n    model.config.use_cache = False\n    model.print_trainable_parameters()\n\n    train_dataset, eval_dataset = load_splits(config, args.train_file, args.val_file)\n\n    train_dataset = train_dataset.map(\n        tokenize_example,\n        fn_kwargs={\"tokenizer\": tokenizer, \"max_seq_length\": model_cfg[\"max_seq_length\"]},\n        remove_columns=train_dataset.column_names,\n        num_proc=config[\"runtime\"][\"dataset_num_proc\"],\n    ).filter(lambda example: not example[\"skip_example\"])\n    eval_dataset = eval_dataset.map(\n        tokenize_example,\n        fn_kwargs={\"tokenizer\": tokenizer, \"max_seq_length\": model_cfg[\"max_seq_length\"]},\n        remove_columns=eval_dataset.column_names,\n        num_proc=config[\"runtime\"][\"dataset_num_proc\"],\n    ).filter(lambda example: not example[\"skip_example\"])\n\n    print(f\"Train rows after tokenization: {len(train_dataset)}\")\n    print(f\"Eval rows after tokenization:  {len(eval_dataset)}\")\n\n    train_args_kwargs = dict(\n        output_dir=str(output_dir / \"checkpoints\"),\n        per_device_train_batch_size=int(train_cfg[\"per_device_train_batch_size\"]),\n        per_device_eval_batch_size=int(train_cfg[\"per_device_eval_batch_size\"]),\n        gradient_accumulation_steps=int(train_cfg[\"gradient_accumulation_steps\"]),\n        learning_rate=float(train_cfg[\"learning_rate\"]),\n        lr_scheduler_type=str(train_cfg[\"lr_scheduler_type\"]),\n        warmup_ratio=float(train_cfg[\"warmup_ratio\"]),\n        weight_decay=float(train_cfg[\"weight_decay\"]),\n        max_grad_norm=float(train_cfg[\"max_grad_norm\"]),\n        num_train_epochs=float(train_cfg[\"num_train_epochs\"]),\n        max_steps=int(train_cfg[\"max_steps\"]) if train_cfg[\"max_steps\"] else -1,\n        logging_steps=int(train_cfg[\"logging_steps\"]),\n        eval_steps=int(train_cfg[\"eval_steps\"]),\n        save_steps=int(train_cfg[\"save_steps\"]),\n        save_total_limit=int(train_cfg[\"save_total_limit\"]),\n        save_strategy=\"steps\",\n        report_to=[] if train_cfg[\"report_to\"] == \"none\" else [train_cfg[\"report_to\"]],\n        bf16=is_bfloat16_supported(),\n        fp16=not is_bfloat16_supported(),\n        optim=str(train_cfg[\"optim\"]),\n        dataloader_num_workers=int(train_cfg[\"dataloader_num_workers\"]),\n        gradient_checkpointing=bool(train_cfg[\"gradient_checkpointing\"]),\n        remove_unused_columns=False,\n        logging_first_step=True,\n        seed=int(config[\"seed\"]),\n    )\n    training_args_signature = inspect.signature(TrainingArguments.__init__)\n    training_args_params = training_args_signature.parameters\n    if \"eval_strategy\" in training_args_params:\n        train_args_kwargs[\"eval_strategy\"] = \"steps\"\n    else:\n        train_args_kwargs[\"evaluation_strategy\"] = \"steps\"\n    if \"group_by_length\" in training_args_params:\n        train_args_kwargs[\"group_by_length\"] = bool(train_cfg[\"group_by_length\"])\n    elif bool(train_cfg[\"group_by_length\"]) and \"train_sampling_strategy\" in training_args_params:\n        train_args_kwargs[\"train_sampling_strategy\"] = \"group_by_length\"\n\n    train_args_kwargs = {\n        key: value\n        for key, value in train_args_kwargs.items()\n        if key in training_args_params\n    }\n\n    train_args = TrainingArguments(**train_args_kwargs)\n\n    trainer_kwargs = dict(\n        model=model,\n        args=train_args,\n        train_dataset=train_dataset,\n        eval_dataset=eval_dataset,\n        data_collator=SupervisedDataCollator(tokenizer),\n    )\n    trainer_signature = inspect.signature(Trainer.__init__)\n    if \"processing_class\" in trainer_signature.parameters:\n        trainer_kwargs[\"processing_class\"] = tokenizer\n    elif \"tokenizer\" in trainer_signature.parameters:\n        trainer_kwargs[\"tokenizer\"] = tokenizer\n\n    trainer = Trainer(**trainer_kwargs)\n    trainer.train()\n\n    if config[\"export\"][\"save_adapter\"]:\n        adapter_dir = output_dir / \"adapter\"\n        model.save_pretrained(str(adapter_dir))\n        tokenizer.save_pretrained(str(adapter_dir))\n\n    if config[\"export\"][\"save_merged_16bit\"]:\n        merged_dir = output_dir / \"merged_16bit\"\n        model.save_pretrained_merged(\n            str(merged_dir),\n            tokenizer,\n            save_method=\"merged_16bit\",\n        )\n\n    if config[\"export\"][\"save_gguf\"]:\n        gguf_dir = output_dir / \"gguf\"\n        model.save_pretrained_gguf(\n            str(gguf_dir),\n            tokenizer,\n            quantization_method=config[\"export\"][\"gguf_quantization_method\"],\n        )\n\n\nif __name__ == \"__main__\":\n    main()\n",
    "scripts/infer_transformers_peft.py": "from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom peft import PeftModel\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\nfrom text2sql_unsloth.config import load_config\nfrom text2sql_unsloth.prompting import build_messages, extract_sql_from_response, render_chat\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Run inference with transformers + PEFT on a trained LoRA adapter.\")\n    parser.add_argument(\"--base-config\", default=\"configs/base.yaml\")\n    parser.add_argument(\"--config\", default=None)\n    parser.add_argument(\"--adapter-dir\", required=True)\n    parser.add_argument(\"--base-model\", default=\"Qwen/Qwen2.5-Coder-7B-Instruct\")\n    parser.add_argument(\"--question\", required=True)\n    parser.add_argument(\"--schema-file\", default=None)\n    parser.add_argument(\"--schema-text\", default=None)\n    parser.add_argument(\"--strategy\", choices=[\"direct_sql\", \"advanced_reasoning\"], default=None)\n    parser.add_argument(\"--max-new-tokens\", type=int, default=256)\n    parser.add_argument(\"--temperature\", type=float, default=0.0)\n    parser.add_argument(\"--top-p\", type=float, default=0.95)\n    return parser.parse_args()\n\n\ndef load_schema(args: argparse.Namespace) -> str:\n    if args.schema_text:\n        return args.schema_text.strip()\n    if args.schema_file:\n        return Path(args.schema_file).read_text(encoding=\"utf-8\").strip()\n    raise ValueError(\"Provide either --schema-file or --schema-text.\")\n\n\ndef load_model(base_model: str, adapter_dir: str) -> tuple[Any, Any]:\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n    tokenizer = AutoTokenizer.from_pretrained(base_model)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"left\"\n\n    model = AutoModelForCausalLM.from_pretrained(\n        base_model,\n        quantization_config=bnb_config,\n        torch_dtype=torch.float16,\n        device_map=\"auto\",\n    )\n    model = PeftModel.from_pretrained(model, adapter_dir)\n    model.eval()\n    return model, tokenizer\n\n\ndef main() -> None:\n    args = parse_args()\n    config = load_config(args.base_config, args.config)\n    schema = load_schema(args)\n    strategy = args.strategy or config[\"prompt\"][\"inference_format\"]\n\n    model, tokenizer = load_model(args.base_model, args.adapter_dir)\n    messages = build_messages(\n        question=args.question,\n        schema=schema,\n        sql=\"\",\n        config=config,\n        include_explanation=False,\n        format_name=strategy,\n    )[:-1]\n    prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)\n    inputs = tokenizer(prompt_text, return_tensors=\"pt\").to(model.device)\n\n    with torch.inference_mode():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=args.max_new_tokens,\n            do_sample=args.temperature > 0,\n            temperature=max(args.temperature, 1e-5),\n            top_p=args.top_p,\n            use_cache=True,\n            pad_token_id=tokenizer.eos_token_id,\n        )\n\n    decoded = tokenizer.decode(output[0][inputs[\"input_ids\"].shape[1]:], skip_special_tokens=True)\n    print(extract_sql_from_response(decoded))\n\n\nif __name__ == \"__main__\":\n    main()\n",
    "scripts/export_model.py": "from __future__ import annotations\n\nimport argparse\n\nfrom text2sql_unsloth.config import load_config\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Export a trained adapter to merged or GGUF artifacts.\")\n    parser.add_argument(\"--base-config\", default=\"configs/base.yaml\")\n    parser.add_argument(\"--config\", default=None)\n    parser.add_argument(\"--adapter-dir\", required=True)\n    parser.add_argument(\"--base-model\", required=True)\n    parser.add_argument(\"--merged-dir\", default=None)\n    parser.add_argument(\"--gguf-dir\", default=None)\n    parser.add_argument(\"--gguf-quant\", default=None)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    config = load_config(args.base_config, args.config)\n    model_cfg = config[\"model\"]\n\n    from unsloth import FastLanguageModel\n\n    model, tokenizer = FastLanguageModel.from_pretrained(\n        model_name=args.adapter_dir,\n        max_seq_length=model_cfg[\"max_seq_length\"],\n        dtype=model_cfg.get(\"dtype\"),\n        load_in_4bit=bool(model_cfg[\"load_in_4bit\"]),\n    )\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    if args.merged_dir:\n        model.save_pretrained_merged(\n            args.merged_dir,\n            tokenizer,\n            save_method=\"merged_16bit\",\n        )\n    if args.gguf_dir:\n        model.save_pretrained_gguf(\n            args.gguf_dir,\n            tokenizer,\n            quantization_method=args.gguf_quant or config[\"export\"][\"gguf_quantization_method\"],\n        )\n\n\nif __name__ == \"__main__\":\n    main()\n",
}

for rel_path, content in FILES.items():
    path = PROJECT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print(f"Wrote {path}")


## Step 3 - Install dependencies

This installs the project dependencies, then refreshes Unsloth using the current Colab-friendly upstream path. The notebook supports current `transformers 5.x` API changes in the synced training script.


In [ ]:
run('nvidia-smi', check=False)
run('python -m pip install -U pip setuptools wheel')
run('python -m pip install -r requirements/colab.txt')
run('python -m pip uninstall -y unsloth unsloth_zoo', check=False)
run("python -m pip install --upgrade --no-cache-dir 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'")
run("python -m pip install --upgrade --no-cache-dir 'git+https://github.com/unslothai/unsloth-zoo.git'")
run('python -m pip install -e .')

if HF_TOKEN:
    login_cmd = f"python -c \"from huggingface_hub import login; login(token={HF_TOKEN!r}, add_to_git_credential=False)\""
    run(login_cmd)


## Step 4 - Generate the runtime config

Change only `DATASET_ROWS` in the first parameter cell to switch between a 20-row smoke test and a 5k run. This cell writes `configs/colab_runtime.yaml` accordingly.


In [ ]:
import yaml

runtime_cfg = {
    'seed': 3407,
    'dataset': {
        'sample_size': int(DATASET_ROWS),
        'output_root': f'data/processed/{RUN_NAME}',
        'split': {
            'train_ratio': 0.80 if DATASET_ROWS <= 20 else 0.90,
            'val_ratio': 0.20 if DATASET_ROWS <= 20 else 0.05,
            'test_ratio': 0.00 if DATASET_ROWS <= 20 else 0.05,
        },
    },
    'model': {
        'base_model_name': BASE_MODEL_NAME,
        'max_seq_length': int(MAX_SEQ_LENGTH),
        'lora': {
            'r': int(LORA_R),
            'alpha': int(LORA_ALPHA),
        },
    },
    'training': {
        'output_dir': f'artifacts/{RUN_NAME}',
        'per_device_train_batch_size': int(TRAIN_BATCH_SIZE),
        'per_device_eval_batch_size': int(EVAL_BATCH_SIZE),
        'gradient_accumulation_steps': int(GRAD_ACCUM),
        'learning_rate': float(LEARNING_RATE),
        'num_train_epochs': int(NUM_EPOCHS),
        'logging_steps': 1 if DATASET_ROWS <= 20 else 5,
        'eval_steps': 5 if DATASET_ROWS <= 20 else 50,
        'save_steps': 5 if DATASET_ROWS <= 20 else 50,
        'save_total_limit': 2,
        'report_to': 'none',
    },
    'export': {
        'save_adapter': True,
        'save_merged_16bit': bool(EXPORT_MERGED_16BIT),
        'save_gguf': False,
    },
}

config_path = Path('configs/colab_runtime.yaml')
config_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding='utf-8')
print(config_path.read_text())


## Step 5 - Inspect the dataset schema

Run this before preprocessing to confirm the Hugging Face dataset still exposes the expected Gretel columns.


In [ ]:
run('python scripts/preprocess_gretel.py --inspect-only')


## Step 6 - Preprocess the Gretel dataset

The preprocessing path keeps read-only SQLite-compatible rows, extracts schema DDL from `sql_context`, deduplicates examples, and writes cleaned splits under the runtime output root.


In [ ]:
preprocess_cmd = 'python scripts/preprocess_gretel.py --config configs/colab_runtime.yaml'
if SPIDER_BLOCKLIST_JSON:
    preprocess_cmd += f' --spider-blocklist-json {SPIDER_BLOCKLIST_JSON}'
run(preprocess_cmd)


In [ ]:
summary = json.loads(Path(f'data/processed/{RUN_NAME}/summary.json').read_text())
train_rows = Path(f'data/processed/{RUN_NAME}/train.jsonl').read_text().splitlines()
first_train = json.loads(train_rows[0])

print(summary)
print('\nPreview example:')
print(json.dumps({
    'question': first_train['sql_prompt'],
    'schema_ddl': first_train['schema_ddl'],
    'target_sql': first_train['sql'],
}, indent=2))


## Step 7 - Train with Unsloth QLoRA

This is the main fine-tuning step. The training script is already patched for current `transformers 5.x` argument changes and imports Unsloth before `transformers`.


In [ ]:
log_dir = Path('logs')
log_dir.mkdir(exist_ok=True)
train_log = log_dir / f'{RUN_NAME}_train.log'
run(f'python -u scripts/train_unsloth.py --config configs/colab_runtime.yaml 2>&1 | tee {train_log}')
print('Training log:', train_log)


In [ ]:
adapter_dir = Path(f'artifacts/{RUN_NAME}/adapter')
assert adapter_dir.exists(), f'Missing adapter directory: {adapter_dir}'
print('Adapter directory:', adapter_dir)
run(f'ls -R artifacts/{RUN_NAME}', check=False)


## Step 8 - Preview post-training inference

This uses a stable `transformers + peft` inference path instead of Unsloth fast inference, which avoids the rotary-cache shape issue seen on some Colab stacks.


In [ ]:
if RUN_PREVIEW_INFERENCE:
    schema_file = Path('/content/colab_preview_schema.sql')
    schema_file.write_text(first_train['schema_ddl'], encoding='utf-8')

    cmd = [
        'python', 'scripts/infer_transformers_peft.py',
        '--config', 'configs/colab_runtime.yaml',
        '--adapter-dir', f'artifacts/{RUN_NAME}/adapter',
        '--base-model', TRANSFORMERS_BASE_MODEL,
        '--question', first_train['sql_prompt'],
        '--schema-file', str(schema_file),
        '--strategy', 'advanced_reasoning',
        '--max-new-tokens', '128',
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print('Question:', first_train['sql_prompt'])
    print('Expected:', first_train['sql'])
    print('Predicted:', result.stdout.strip())
    if result.stderr.strip():
        print('stderr:', result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Inference failed with code {result.returncode}')
else:
    print('Skipping inference preview because RUN_PREVIEW_INFERENCE is False.')


## Step 9 - Optional merged-model export

Leave `EXPORT_MERGED_16BIT = False` on smaller Colab GPUs. Turn it on only if you need a merged HF model and have enough VRAM and disk.


In [ ]:
if EXPORT_MERGED_16BIT:
    run(
        'python -u scripts/export_model.py '
        '--config configs/colab_runtime.yaml '
        f'--adapter-dir artifacts/{RUN_NAME}/adapter '
        f'--base-model {BASE_MODEL_NAME} '
        f'--merged-dir artifacts/{RUN_NAME}/merged_16bit'
    )
    print('Merged model export completed.')
else:
    print('Skipping merged-model export.')


## Step 10 - Zip and optionally copy the artifacts

This always packages the adapter. If Drive saving is enabled, it also copies the runtime config, summary, logs, and adapter zip to Drive.


In [ ]:
import shutil

adapter_zip_base = f'/content/{RUN_NAME}_adapter'
adapter_zip_path = shutil.make_archive(adapter_zip_base, 'zip', root_dir=f'artifacts/{RUN_NAME}', base_dir='adapter')
print('Adapter zip:', adapter_zip_path)

bundle_dir = Path(f'/content/{RUN_NAME}_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2('configs/colab_runtime.yaml', bundle_dir / 'colab_runtime.yaml')
shutil.copy2(f'data/processed/{RUN_NAME}/summary.json', bundle_dir / 'summary.json')
shutil.copy2(f'logs/{RUN_NAME}_train.log', bundle_dir / f'{RUN_NAME}_train.log')
shutil.copy2(adapter_zip_path, bundle_dir / Path(adapter_zip_path).name)

if EXPORT_MERGED_16BIT and Path(f'artifacts/{RUN_NAME}/merged_16bit').exists():
    merged_note = bundle_dir / 'merged_model_path.txt'
    merged_note.write_text(str(Path(f'artifacts/{RUN_NAME}/merged_16bit').resolve()), encoding='utf-8')

bundle_zip_path = shutil.make_archive(f'/content/{RUN_NAME}_bundle', 'zip', root_dir=bundle_dir.parent, base_dir=bundle_dir.name)
print('Bundle zip:', bundle_zip_path)

if SAVE_TO_DRIVE:
    drive_root = Path(DRIVE_OUTPUT_ROOT) / RUN_NAME
    drive_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(adapter_zip_path, drive_root / Path(adapter_zip_path).name)
    shutil.copy2(bundle_zip_path, drive_root / Path(bundle_zip_path).name)
    print('Copied artifacts to:', drive_root)


## Next use

- For a smoke test, leave `DATASET_ROWS = 20` and confirm training plus inference work.
- For a scale-up run, change `DATASET_ROWS = 5000` and rerun from the parameter cell downward.
- If you only want to rerun training with a different row count, rerun from the parameter cell, then the runtime-config cell, preprocessing, and training.
- Keep the adapter zip. That is the main artifact you will later use on Vast.ai or for export.
